In [3]:

!pip install numpy pandas matplotlib mne scikit-learn


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import os
import re
from sklearn.model_selection import train_test_split
from collections import Counter


In [5]:
# Folder path (make sure 'chb_mit' is in same directory as this notebook)
data_dir = './chb_mit/'  # ✅ Corrected folder name

# Sampling and windowing setup
sampling_rate = 256       # CHB-MIT default
window_size_sec = 10      # 10-second windows
stride_sec = 5            # 5-second stride
window_size = window_size_sec * sampling_rate
stride = stride_sec * sampling_rate


In [6]:
def parse_all_summaries(data_dir):
    seizure_dict = {}
    for file in os.listdir(data_dir):
        if file.endswith('-summary.txt'):
            with open(os.path.join(data_dir, file), 'r') as f:
                lines = f.readlines()

            current_file = None
            start_time = None

            for line in lines:
                if '.edf' in line:
                    match = re.search(r'File Name: (chb\d+_\d+\.edf)', line)
                    if match:
                        current_file = match.group(1)
                        seizure_dict[current_file] = []

                elif 'Seizure' in line and 'Start Time' in line:
                    start_match = re.search(r'Start Time: (\d+)', line)
                    if start_match:
                        start_time = int(start_match.group(1))

                elif 'Seizure' in line and 'End Time' in line:
                    end_match = re.search(r'End Time: (\d+)', line)
                    if end_match and start_time is not None and current_file:
                        end_time = int(end_match.group(1))
                        seizure_dict[current_file].append((start_time, end_time))
                        start_time = None
    return seizure_dict


In [7]:
def segment_and_label(data, seizure_times):
    segments, labels = [], []
    for start in range(0, data.shape[1] - window_size, stride):
        end = start + window_size
        segment = data[:, start:end]
        seg_start_sec = start / sampling_rate
        seg_end_sec = end / sampling_rate

        label = 0
        for sz_start, sz_end in seizure_times:
            if seg_start_sec < sz_end and seg_end_sec > sz_start:
                label = 1
                break

        segments.append(segment)
        labels.append(label)
    return segments, labels


In [8]:
# Load summary annotations and EDF file list
seizure_annotations = parse_all_summaries(data_dir)
edf_files = [f for f in os.listdir(data_dir) if f.endswith('.edf')]

print(f"📄 Total summary files parsed: {len(seizure_annotations)}")
print(f"📁 Found EDF files: {len(edf_files)}")


📄 Total summary files parsed: 654
📁 Found EDF files: 686


In [9]:
all_segments = []
all_labels = []

for edf_file in edf_files:
    edf_path = os.path.join(data_dir, edf_file)
    print(f"🔄 Processing {edf_file}...")

    try:
        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        raw.crop(tmax=600)
        raw.load_data()

        bad_channels = [ch for ch in raw.ch_names if ch.strip() == '' or ch.startswith('--') or ch == '-']
        if bad_channels:
            raw.drop_channels(bad_channels)

        if raw.n_times < window_size:
            print(f"⏩ Skipped {edf_file}: too short ({raw.n_times} < {window_size})")
            continue

        raw.filter(1., 70., fir_design='firwin')
        raw.notch_filter(freqs=60)

        data, _ = raw.get_data(return_times=True)

    except Exception as e:
        print(f"❌ Failed to load {edf_file}: {e}")
        continue

    seizure_times = seizure_annotations.get(edf_file, [])
    segments, labels = segment_and_label(data, seizure_times)

    valid_count = 0
    for s, l in zip(segments, labels):
        if s.shape == (data.shape[0], window_size):
            all_segments.append(s)
            all_labels.append(l)
            valid_count += 1
        else:
            print(f"⚠ Skipped segment with shape {s.shape} from {edf_file}")

    print(f"✅ Processed {edf_file} | Valid Segments: {valid_count}")


🔄 Processing chb01_01.edf...
Reading 0 ... 153600  =      0.000 ...   600.000 secs...


C:\Users\Sukanya\AppData\Local\Temp\ipykernel_21548\1525084356.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 845 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


✅ Processed chb01_01.edf | Valid Segments: 119
🔄 Processing chb01_02.edf...
Reading 0 ... 153600  =      0.000 ...   600.000 secs...


...

✅ Processed chb24_21.edf | Valid Segments: 119
🔄 Processing chb24_22.edf...
Reading 0 ... 153600  =      0.000 ...   600.000 secs...


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s
C:\Users\Sukanya\AppData\Local\Temp\ipykernel_21548\1525084356.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 70 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 70.00 Hz
- Upper transition bandwidth: 17.50 Hz (-6 dB cutoff frequency: 78.75 Hz)
- Filter length: 845 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.2s


✅ Processed chb24_22.edf | Valid Segments: 119


In [10]:
import numpy as np
import os
from collections import Counter

# ✅ Step 1: Recreate batches folder
os.makedirs("batches", exist_ok=True)
print("📁 Ready: 'batches/' folder created.")

# ✅ Step 2: Find most common shape among segments
shapes = [s.shape for s in all_segments]
common_shape, count = Counter(shapes).most_common(1)[0]
print(f"📐 Most common shape: {common_shape} ({count} samples)")

# ✅ Step 3: Filter only valid segments and labels
filtered_segments = [s.astype(np.float32) for s in all_segments if s.shape == common_shape]
filtered_labels = [l for s, l in zip(all_segments, all_labels) if s.shape == common_shape]

print(f"✅ Total valid samples: {len(filtered_segments)}")

# ✅ Step 4: Save in clean .npz batches
batch_size = 1000

for i in range(0, len(filtered_segments), batch_size):
    X_batch = np.stack(filtered_segments[i:i+batch_size])  # (B, 23, 2560)
    y_batch = np.array(filtered_labels[i:i+batch_size], dtype=np.int32)

    filename = f"batches/batch_{i // batch_size}.npz"
    np.savez_compressed(filename, X=X_batch, y=y_batch)

    print(f"📦 Saved {filename} → X: {X_batch.shape}, y: {y_batch.shape}")

print("✅ All batches saved successfully.")


📁 Ready: 'batches/' folder created.
📐 Most common shape: (23, 2560) (57834 samples)
✅ Total valid samples: 57834
📦 Saved batches/batch_0.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_1.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_2.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_3.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_4.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_5.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_6.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_7.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_8.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_9.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_10.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_11.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_12.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches/batch_13.npz → X: (1000, 23, 2560), y: (1000,)
📦 Saved batches

In [12]:
import numpy as np
import os

label_counts = {0: 0, 1: 0}
for fname in os.listdir("batches"):
    path = os.path.join("batches", fname)
    data = np.load(path)
    labels = data["y"]
    unique, counts = np.unique(labels, return_counts=True)
    for u, c in zip(unique, counts):
        label_counts[u] += c

print("📊 Total label distribution across all batches:")
for label, count in label_counts.items():
    print(f"  Label {label}: {count} samples")


📊 Total label distribution across all batches:
  Label 0: 57673 samples
  Label 1: 161 samples


In [11]:
import numpy as np

# Load one batch file to test
sample_batch = np.load("batches/batch_0.npz")

X_sample = sample_batch["X"]  # (1000, 23, 2560)
y_sample = sample_batch["y"]  # (1000,)

# Show details
print("✅ X_sample shape:", X_sample.shape)
print("✅ y_sample shape:", y_sample.shape)
print("✅ Data type:", X_sample.dtype)
print("✅ First 5 labels:", y_sample[:5])


✅ X_sample shape: (1000, 23, 2560)
✅ y_sample shape: (1000,)
✅ Data type: float32
✅ First 5 labels: [0 0 0 0 0]
